# Redis para documentos técnicos — demonstração no Google Colab

Implementação individual de banco NoSQL: **Redis** (chave-valor em memória) com backend **Django**.

Este notebook sobe um Redis dentro do próprio Colab, carrega o projeto e executa o fluxo completo de um documento técnico:
criar → submeter para a fila → retirar da fila → validar → histórico → cache com TTL → ranking.

**Como usar:** `Ambiente de execução › Executar tudo` (ou `Ctrl+F9`). Na gravação, rode célula por célula.

> O Colab é temporário: ao reiniciar o ambiente, o Redis e os dados somem. É só executar tudo de novo.

## 1. Preparar o ambiente
Instala e inicia o servidor Redis, e instala Django e o cliente `redis-py`.

In [ ]:
!apt-get -qq update > /dev/null
!apt-get -qq install -y redis-server > /dev/null
!redis-server --daemonize yes --appendonly yes --dir /tmp
!pip -q install "Django>=5.0,<6.0" "redis>=5.0,<6.0"
!sleep 1 && redis-cli ping

## 2. Carregar o projeto

Se o repositório já estiver público no GitHub, coloque a URL abaixo.
Se ainda não estiver, deixe como está: o notebook vai pedir o upload do `redis-docs-tecnicos.zip`.

In [ ]:
REPO_URL = "https://github.com/yzvictorr/redis-docs-tecnicos.git"  # <- troque pela URL do seu repositório

import os, shutil, subprocess, zipfile

PASTA = "/content/redis-docs-tecnicos"
if not os.path.exists(os.path.join(PASTA, "manage.py")):
    if "yzvictorr" not in REPO_URL:
        subprocess.run(["git", "clone", "-q", REPO_URL, PASTA], check=True)
    else:
        from google.colab import files
        print("Selecione o arquivo redis-docs-tecnicos.zip")
        enviado = files.upload()
        nome = next(iter(enviado))
        with zipfile.ZipFile(nome) as z:
            z.extractall("/content")
print("Projeto em", PASTA, "->", sorted(os.listdir(PASTA)))

In [ ]:
import sys, django

os.chdir(PASTA)
if PASTA not in sys.path:
    sys.path.insert(0, PASTA)
os.environ["DJANGO_SETTINGS_MODULE"] = "config.settings"
os.environ["REDIS_URL"] = "redis://localhost:6379/0"
os.environ["RESUMO_TTL_SEGUNDOS"] = "60"
os.environ["RESUMO_ATRASO_SEGUNDOS"] = "1.5"
django.setup()

from documentos.redis_client import get_redis
r = get_redis()
print("Django", django.get_version(), "| Redis", r.info("server")["redis_version"], "| PING:", r.ping())

## 3. Funções de apoio

`api()` chama os endpoints do Django com o *test client* (não precisa de servidor web aberto)
e `ver_chaves()` mostra o que está guardado no Redis, com o tipo de cada chave.

In [ ]:
import json, time
from django.test import Client
from IPython.display import display, Markdown

cliente = Client()

def api(metodo, caminho, corpo=None):
    inicio = time.perf_counter()
    funcao = getattr(cliente, metodo.lower())
    if corpo is not None:
        resp = funcao(caminho, json.dumps(corpo), content_type="application/json")
    else:
        resp = funcao(caminho)
    ms = (time.perf_counter() - inicio) * 1000
    print(f"{metodo.upper()} {caminho}  ->  HTTP {resp.status_code}  ({ms:.1f} ms)")
    if resp.content:
        dados = resp.json()
        print(json.dumps(dados, indent=2, ensure_ascii=False))
        return dados

def ver_chaves(padrao="*"):
    linhas = ["| Chave | Tipo | Tamanho / valor | TTL |", "|---|---|---|---|"]
    for chave in sorted(r.scan_iter(padrao)):
        tipo = r.type(chave)
        tamanho = {
            "hash": lambda: f"{r.hlen(chave)} campos",
            "list": lambda: f"{r.llen(chave)} itens",
            "set": lambda: f"{r.scard(chave)} membros",
            "zset": lambda: f"{r.zcard(chave)} membros",
            "string": lambda: (r.get(chave) or "")[:40],
        }[tipo]()
        ttl = r.ttl(chave)
        linhas.append(f"| `{chave}` | {tipo.upper()} | {tamanho} | {'—' if ttl == -1 else str(ttl) + ' s'} |")
    display(Markdown("\n".join(linhas)))

## 4. Popular com dados de exemplo
Cria 8 documentos técnicos, coloca parte na fila, valida alguns e simula acessos para o ranking.

In [ ]:
from django.core.management import call_command
call_command("seed", "--limpar")
ver_chaves()

## 5. Demonstração

### 5.1 Criar um documento
O ID vem de um `INCR` (atômico). O documento é um `HASH`, e os índices são gravados na mesma transação `MULTI/EXEC`.

In [ ]:
doc = api("POST", "/api/documentos/", {
    "titulo": "Especificação de rebites - fuselagem traseira",
    "tipo": "especificacao",
    "autor": "Engenharia",
    "conteudo": "Define o tipo de rebite, o diâmetro, o passo e o procedimento de instalação "
                "para os painéis da fuselagem traseira, com critérios de inspeção visual e dimensional.",
})
ID = doc["id"]
print("\nNo Redis:", r.hgetall(f"doc:{ID}"))

### 5.2 Submeter para a fila de validação
`LPUSH` coloca o ID no fim da fila; `SMOVE` muda o documento de índice de status.

In [ ]:
api("POST", f"/api/documentos/{ID}/submeter/")
api("GET", "/api/fila/")
print("\nfila:validacao =", r.lrange("fila:validacao", 0, -1), "(o da direita sai primeiro)")

### 5.3 O validador pega o próximo
`RPOP` retira o mais antigo: o primeiro que entrou é o primeiro que sai (FIFO).
Repare que sai um documento do *seed*, que estava esperando antes, e não o que acabamos de enviar.

In [ ]:
api("POST", "/api/fila/proximo/")
print("\nfila:validacao agora =", r.lrange("fila:validacao", 0, -1))

### 5.4 Validar (reprovar com comentário) e ver o histórico
A troca de status usa `WATCH` + `MULTI/EXEC`. O histórico é uma `LIST` limitada a 50 eventos com `LTRIM`.

In [ ]:
api("POST", f"/api/documentos/{ID}/validar/", {"aprovado": False, "comentario": "Faltam os registros de calibração"})
api("GET", f"/api/documentos/{ID}/historico/")

Tentativa inválida: aprovar um documento que já foi reprovado. O sistema recusa com **HTTP 409**.

In [ ]:
api("POST", f"/api/documentos/{ID}/validar/", {"aprovado": True})

### 5.5 Cache com TTL
Primeira chamada: calcula (simula a IA, ~1,5 s) e grava com `SET ... EX 60`. Segunda chamada: vem do cache.

In [ ]:
primeira = api("GET", f"/api/documentos/{ID}/resumo/")
print()
segunda = api("GET", f"/api/documentos/{ID}/resumo/")
print(f"\n{primeira['fonte']}: {primeira['tempo_resposta_ms']} ms  |  {segunda['fonte']}: {segunda['tempo_resposta_ms']} ms")
print("TTL restante da chave:", r.ttl(f"cache:resumo:{ID}"), "s")

Editar o documento **invalida o cache** (`DEL`) e sobe a versão (`HINCRBY`):

In [ ]:
api("PATCH", f"/api/documentos/{ID}/", {"conteudo": "Conteúdo revisado com os registros de calibração anexados."})
print("\nchave de cache existe?", bool(r.exists(f"cache:resumo:{ID}")))

### 5.6 Ranking dos mais acessados
Cada leitura faz `ZINCRBY`; o ranking sai pronto com `ZREVRANGE ... WITHSCORES`.

In [ ]:
for _ in range(20):
    cliente.get(f"/api/documentos/{ID}/")
api("GET", "/api/ranking/?n=5")

### 5.7 Estatísticas e visão geral das chaves

In [ ]:
api("GET", "/api/estatisticas/")
ver_chaves()

### 5.8 Os mesmos dados direto no Redis (`redis-cli`)

In [ ]:
!redis-cli ZREVRANGE ranking:acessos 0 4 WITHSCORES
!redis-cli SMEMBERS docs:status:em_validacao
!redis-cli LRANGE fila:validacao 0 -1

## 6. Testes automatizados
Rodam contra o Redis real, no *database* 15, sem mexer nos dados da demonstração.

In [ ]:
!cd /content/redis-docs-tecnicos && REDIS_URL=redis://localhost:6379/0 python manage.py test -v 2